# Multi-Model Forecasting Pipeline with Checkpoints

This notebook works with the existing `parse_data.py` and `training.py` files **without editing them**.

Important fix:
- We do **not** call `parse_dataset()` because the current `parse_data.py` calls `load_dataset_main()`, but `load_dataset_main()` is not defined in that file.
- Instead, we call only functions that actually exist:
  - `load_dataset`
  - `split_by_subject`
  - `normalize_items`
  - `build_sliding_windows`

Mentor-requested changes:
1. Save model checkpoints every 10 epochs for PyTorch models.
2. Save each model's result right after it finishes.
3. Merge all saved result files at the end.

Models included:
1. Linear Regression
2. LSTM
3. Transformer

Exponential smoothing is skipped for now.


In [1]:
# Cell 1: Basic imports

import os
import sys
import time
import json
import joblib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# Make sure notebook can import local repo modules
repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

print("Current folder:", repo_root)
print("Python executable:", sys.executable)
print("PyTorch version:", torch.__version__)


Current folder: /home/oz6/projects/fmri_forecasting
Python executable: /home/oz6/micromamba/envs/fmri/bin/python
PyTorch version: 2.6.0+cu124


In [2]:
# Cell 2: Import only functions/classes that actually exist

# From parse_data.py
from utils.parse_data import (
    load_dataset,
    split_by_subject,
    normalize_items,
    build_sliding_windows,
)

# From training.py
from utils.training import (
    FMRIWindowDataset,
    DeltaAwareLoss,
    train_forecasting_model,
    predict_forecasting_model,
    compute_rmse,
    compute_naive_rmse,
    compute_rmsse,
    compute_eta,
    horizon_rmse,
)

print("Existing parser/training functions imported successfully.")


Existing parser/training functions imported successfully.


In [3]:
# Cell 3: Import model generators

from models.linear_regression.linear_regression_core import linear_regression_generator
from models.lstm.lstm_model_library import alstm_model_generator
from models.transformer.transformer_api_library import transformer_model_generator

print("Model generators imported successfully.")


Model generators imported successfully.


In [4]:
# Cell 4: Paths and training settings

ROOT_DIR = Path("data") / "pooled_stratified_share"

M = 50
H = 3
STRIDE = 1
TEST_RATIO = 0.2
RANDOM_STATE = 42

NUM_EPOCHS = 50
BATCH_SIZE = 512
CHECKPOINT_EVERY = 10

RESULTS_DIR = Path("results")
CHECKPOINT_DIR = Path("checkpoints")

RESULTS_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR.mkdir(exist_ok=True)

print("Dataset folder:", ROOT_DIR)
print("Dataset exists?:", ROOT_DIR.exists())
print("Results folder:", RESULTS_DIR.resolve())
print("Checkpoint folder:", CHECKPOINT_DIR.resolve())


Dataset folder: data/pooled_stratified_share
Dataset exists?: True
Results folder: /home/oz6/projects/fmri_forecasting/results
Checkpoint folder: /home/oz6/projects/fmri_forecasting/checkpoints


## Parse data manually using existing functions

This replaces `parse_dataset()`.

The order is:

```text
load raw dataset
→ split subjects
→ normalize train/test separately
→ build train/test sliding windows
```

This keeps the test subjects fully held out and avoids train/test leakage.


In [ ]:
# Cell 5: Data Parsing

if not ROOT_DIR.exists():
    raise FileNotFoundError(f"Dataset folder not found: {ROOT_DIR}")

# Load Raw Dataset
print("Loading raw dataset...")
start = time.time()
dataset = load_dataset(ROOT_DIR)
print(f"Loaded {len(dataset)} runs in {time.time() - start:.2f} seconds")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Splitting Data by Subject
print("Splitting dataset by subject...")
train_items, test_items = split_by_subject(
    dataset,
    test_ratio=TEST_RATIO,
    test_subjects=None,
    random_state=RANDOM_STATE,
    verbose=True,
)

# Normalizing
print("Normalizing train and test groups separately...")
train_items = normalize_items(train_items)
test_items = normalize_items(test_items)

# Sliding windows
print("Building train windows...")
X_train, Y_train = build_sliding_windows(
    train_items,
    M=M,
    H=H,
    stride=STRIDE,
)

# Test Windows
print("Building test windows...")
X_test, Y_test = build_sliding_windows(
    test_items,
    M=M,
    H=H,
    stride=STRIDE,
)

print("Final shapes:")
print("X_train:", X_train.shape)
print("Y_train:", Y_train.shape)
print("X_test :", X_test.shape)
print("Y_test :", Y_test.shape)


Loading raw dataset...
Scanning and loading dataset (optimized fixed schema mode)...
Loaded: 2144 runs from 6 subjects
Subjects: ['subjOenz4SHyaO', 'subjQX6R7rImeb', 'subjoVC0sHUB_P', 'subjqFdsBbuiHF', 'subjtySNgRhV65', 'subjxpYwO4azeZ']
Loaded 2144 runs in 53.74 seconds
Device: cuda
Splitting dataset by subject...
Train subjects: ['subjqFdsBbuiHF', 'subjoVC0sHUB_P', 'subjxpYwO4azeZ', 'subjtySNgRhV65']
Test subjects : ['subjQX6R7rImeb', 'subjOenz4SHyaO']
Train runs: 1440
Test runs : 704
Normalizing train and test groups separately...
Building train windows...
Building test windows...
Final shapes:
X_train: (250560, 50, 19)
Y_train: (250560, 3, 19)
X_test : (122496, 50, 19)
Y_test : (122496, 3, 19)


In [8]:
# Cell 6: Sanity checks

assert len(X_train) > 0, "X_train is empty. Check M, H, stride, or train data length."
assert len(X_test) > 0, "X_test is empty. Check M, H, stride, or test data length."

assert X_train.ndim == 3, "X_train should be (N_train, M, ROI)"
assert Y_train.ndim == 3, "Y_train should be (N_train, H, ROI)"
assert X_test.ndim == 3, "X_test should be (N_test, M, ROI)"
assert Y_test.ndim == 3, "Y_test should be (N_test, H, ROI)"

assert X_train.shape[1] == M
assert Y_train.shape[1] == H
assert X_train.shape[2] == Y_train.shape[2]
assert X_test.shape[2] == Y_test.shape[2]

n_roi = X_train.shape[2]

print("Sanity checks passed.")
print("Number of ROIs:", n_roi)


Sanity checks passed.
Number of ROIs: 19


## Helper functions for this notebook

These are notebook-level wrappers. We are **not** editing `training.py`.

Main purpose:
- train PyTorch models with checkpoint saving every 10 epochs
- evaluate any model
- save one CSV immediately after each model
- merge results at the end


In [ ]:
# Cell 7: Helper functions for checkpoints and evaluation

def safe_name(name):
    """Turn model name into a file-friendly string."""
    return name.lower().replace(" ", "_").replace("/", "_")


def is_torch_model(model):
    """Check whether model is a PyTorch model."""
    return isinstance(model, nn.Module)


def save_model_checkpoint(model, optimizer, epoch, model_name, train_loss, val_loss=None):
    ckpt_path = CHECKPOINT_DIR / f"{safe_name(model_name)}_epoch_{epoch:03d}.pt"

    checkpoint = {
        "model_name": model_name,
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss,
        "M": M,
        "H": H,
        "stride": STRIDE,
        "n_roi": n_roi,
    }

    torch.save(checkpoint, ckpt_path)
    print(f"  Checkpoint saved: {ckpt_path}")


def train_torch_model_with_checkpoints(
    model,
    X_train,
    Y_train,
    model_name,
    X_val=None,
    Y_val=None,
    num_epochs=50,
    batch_size=512,
    device=device,
    checkpoint_every=10,
):

    model = model.to(device)

    train_loader = DataLoader(
        FMRIWindowDataset(X_train, Y_train),
        batch_size=batch_size,
        shuffle=True,
        pin_memory=True,
    )

    val_loader = None
    if X_val is not None and Y_val is not None and len(X_val) > 0:
        val_loader = DataLoader(
            FMRIWindowDataset(X_val, Y_val),
            batch_size=batch_size,
            shuffle=False,
            pin_memory=True,
        )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=5e-4,
        weight_decay=1e-5,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=2,
    )

    criterion = DeltaAwareLoss(alpha=0.3, delta=0.5)

    best_val_loss = float("inf")
    best_state = None

    for epoch in range(1, num_epochs + 1):

        model.train()
        total_loss = 0.0

        for X_batch, Y_batch in train_loader:
            X_batch = X_batch.to(device, non_blocking=True)
            Y_batch = Y_batch.to(device, non_blocking=True)

            optimizer.zero_grad()

            preds = model(X_batch)
            loss = criterion(preds, Y_batch)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_train_loss = total_loss / max(len(train_loader), 1)

        avg_val_loss = None
        if val_loader is not None:
            model.eval()
            val_loss_total = 0.0

            with torch.no_grad():
                for X_batch, Y_batch in val_loader:
                    X_batch = X_batch.to(device, non_blocking=True)
                    Y_batch = Y_batch.to(device, non_blocking=True)

                    preds = model(X_batch)
                    loss = criterion(preds, Y_batch)
                    val_loss_total += loss.item()

            avg_val_loss = val_loss_total / max(len(val_loader), 1)
            scheduler.step(avg_val_loss)

            print(
                f"Epoch {epoch:03d}/{num_epochs} | "
                f"train: {avg_train_loss:.6f} | "
                f"val: {avg_val_loss:.6f}"
            )

            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                best_state = {
                    k: v.detach().cpu().clone()
                    for k, v in model.state_dict().items()
                }

        else:
            scheduler.step(avg_train_loss)

            print(
                f"Epoch {epoch:03d}/{num_epochs} | "
                f"train: {avg_train_loss:.6f}"
            )

        # Save every 10 epoches
        if epoch % checkpoint_every == 0:
            save_model_checkpoint(
                model=model,
                optimizer=optimizer,
                epoch=epoch,
                model_name=model_name,
                train_loss=avg_train_loss,
                val_loss=avg_val_loss,
            )

    # Restore best validation weights if validation was used.
    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"Best validation weights restored. Best val loss: {best_val_loss:.6f}")

    # Save final model too, so we always have the final trained weights.
    final_path = CHECKPOINT_DIR / f"{safe_name(model_name)}_final.pt"
    torch.save(
        {
            "model_name": model_name,
            "model_state_dict": model.state_dict(),
            "M": M,
            "H": H,
            "stride": STRIDE,
            "n_roi": n_roi,
        },
        final_path,
    )
    print(f"Final PyTorch model saved: {final_path}")

    return model


def evaluate_and_save_model(model_name, model, X_test, Y_test, X_train):
    """
    Predict on test data, compute metrics, and save this model's result immediately.
    """

    preds, targets = predict_forecasting_model(
        model=model,
        X=X_test,
        Y=Y_test,
        batch_size=BATCH_SIZE,
        device=device,
    )

    model_rmse = compute_rmse(targets, preds)
    naive_rmse = compute_naive_rmse(X_test, Y_test)
    model_rmsse = compute_rmsse(targets, preds, X_train)
    eta = compute_eta(targets, preds)
    horizon_scores = horizon_rmse(targets, preds)

    result = {
        "model": model_name,
        "M": M,
        "H": H,
        "stride": STRIDE,
        "num_epochs": NUM_EPOCHS if is_torch_model(model) else None,
        "model_rmse": model_rmse,
        "naive_rmse": naive_rmse,
        "rmsse": model_rmsse,
        "eta": eta,
        "beat_naive": model_rmse < naive_rmse,
        "horizon_rmse": json.dumps([float(x) for x in horizon_scores]),
    }

    result_df = pd.DataFrame([result])

    result_path = RESULTS_DIR / f"{safe_name(model_name)}_results.csv"
    result_df.to_csv(result_path, index=False)

    print(f"Saved result for {model_name}: {result_path}")

    return result, preds, targets


In [10]:
# Cell 8: Build train/validation split for PyTorch models

# We keep the original X_test/Y_test untouched.
# This validation split is only taken from training windows.
# It is used for monitoring neural net training.

val_split_idx = int(len(X_train) * 0.9)

X_tr = X_train[:val_split_idx]
Y_tr = Y_train[:val_split_idx]

X_val = X_train[val_split_idx:]
Y_val = Y_train[val_split_idx:]

print("Training subset:", X_tr.shape, Y_tr.shape)
print("Validation subset:", X_val.shape, Y_val.shape)
print("Held-out test :", X_test.shape, Y_test.shape)


Training subset: (225504, 50, 19) (225504, 3, 19)
Validation subset: (25056, 50, 19) (25056, 3, 19)
Held-out test : (122496, 50, 19) (122496, 3, 19)


## Model configs

Linear Regression trains quickly and does not need epoch checkpoints.

LSTM and Transformer are PyTorch models, so they use the custom checkpoint training loop above.


In [ ]:
# Cell 9: Build models

model_configs = [
    {
        "name": "Linear Regression",
        "model": linear_regression_generator(alpha=1.0),
    },
    {
        "name": "LSTM",
        "model": alstm_model_generator(
            n_roi=n_roi,
            H=H,
        ),
    },
    {
        "name": "Transformer",
        "model": transformer_model_generator(
            n_roi=n_roi,
            M=M,
            H=H,
            d_model=64,
            nhead=4,
            num_layers=2,
            dropout=0.1,
        ),
        
    },
]

print("Models built:")
for cfg in model_configs:
    print(" -", cfg["name"], "| torch?", is_torch_model(cfg["model"]))


Models built:
 - Linear Regression | torch? False
 - LSTM | torch? True
 - Transformer | torch? True


In [13]:
# Cell 10: Train and evaluate all models

all_results = []
trained_models = {}
last_predictions = {}

for cfg in model_configs:

    model_name = cfg["name"]
    model = cfg["model"]

    print(f"Starting model: {model_name}")
    print("=" * 80)

    start_time = time.time()

    if is_torch_model(model):

        # PyTorch models get checkpoints every 10 epochs
        model = train_torch_model_with_checkpoints(
            model=model,
            X_train=X_tr,
            Y_train=Y_tr,
            X_val=X_val,
            Y_val=Y_val,
            model_name=model_name,
            num_epochs=NUM_EPOCHS,
            batch_size=BATCH_SIZE,
            device=device,
            checkpoint_every=CHECKPOINT_EVERY,
        )

    else:

        # Sklearn-style models use the existing training.py helper.
        model = train_forecasting_model(
            model=model,
            X_train=X_train,
            Y_train=Y_train,
            device=device,
        )

        # Save sklearn-style model immediately.
        model_path = CHECKPOINT_DIR / f"{safe_name(model_name)}_model.joblib"
        joblib.dump(model, model_path)
        print(f"Saved sklearn-style model: {model_path}")

    elapsed = time.time() - start_time
    print(f"Finished training {model_name} in {elapsed / 60:.2f} minutes")

    # Saving after each model
    result, preds, targets = evaluate_and_save_model(
        model_name=model_name,
        model=model,
        X_test=X_test,
        Y_test=Y_test,
        X_train=X_train,
    )

    result["train_time_minutes"] = elapsed / 60

    pd.DataFrame([result]).to_csv(
        RESULTS_DIR / f"{safe_name(model_name)}_results.csv",
        index=False,
    )

    all_results.append(result)
    trained_models[model_name] = model
    last_predictions[model_name] = {
        "preds": preds,
        "targets": targets,
    }

    print("Main metrics:")
    print(f"Model RMSE : {result['model_rmse']:.6f}")
    print(f"Naive RMSE : {result['naive_rmse']:.6f}")
    print(f"RMSSE      : {result['rmsse']:.6f}")
    print(f"Eta        : {result['eta']:.6f}")
    print(f"Beat naive?: {'YES' if result['beat_naive'] else 'NO'}")


Starting model: Linear Regression
Saved sklearn-style model: checkpoints/linear_regression_model.joblib
Finished training Linear Regression in 0.02 minutes

Horizon-wise RMSE:
  Step 1 RMSE: 0.701869
  Step 2 RMSE: 0.825718
  Step 3 RMSE: 0.861593
Saved result for Linear Regression: results/linear_regression_results.csv
Main metrics:
Model RMSE : 0.799327
Naive RMSE : 0.991941
RMSSE      : 1.007974
Eta        : 0.122541
Beat naive?: YES
Starting model: LSTM
Epoch 001/50 | train: 0.294751 | val: 0.264470
Epoch 002/50 | train: 0.274939 | val: 0.260088
Epoch 003/50 | train: 0.269107 | val: 0.258207
Epoch 004/50 | train: 0.266244 | val: 0.257073
Epoch 005/50 | train: 0.264113 | val: 0.256899
Epoch 006/50 | train: 0.262140 | val: 0.256952
Epoch 007/50 | train: 0.260134 | val: 0.257875
Epoch 008/50 | train: 0.258099 | val: 0.258643
Epoch 009/50 | train: 0.254358 | val: 0.259756
Epoch 010/50 | train: 0.252628 | val: 0.261204
  Checkpoint saved: checkpoints/lstm_epoch_010.pt
Epoch 011/50 | tra

In [ ]:
# Cell 11: Merge all per-model result files

result_files = sorted(RESULTS_DIR.glob("*_results.csv"))

print("Found result files:")
for f in result_files:
    print(" -", f)

merged_results = pd.concat(
    [pd.read_csv(f) for f in result_files],
    ignore_index=True,
)

merged_path = RESULTS_DIR / "all_model_results_merged.csv"
merged_results.to_csv(merged_path, index=False)

print("Merged results saved to:", merged_path)

merged_results

Found result files:
 - results/linear_regression_results.csv
 - results/lstm_results.csv
 - results/transformer_results.csv
Merged results saved to: results/all_model_results_merged.csv


,model,M,H,stride,num_epochs,model_rmse,naive_rmse,rmsse,eta,beat_naive,horizon_rmse,train_time_minutes
0,Linear Regression,50,3,1,NaN,0.799327,0.991941,1.007974,0.122541,True,"[0.7018686532974243, 0.8257179260253906, 0.861...",0.016238
1,LSTM,50,3,1,50.0,0.807161,0.991941,1.017967,0.114787,True,"[0.7087351679801941, 0.8346214890480042, 0.869...",25.079428
2,Transformer,50,3,1,50.0,0.799953,0.991941,1.008278,0.122251,True,"[0.7018437385559082, 0.8267557621002197, 0.862...",18.700174


: 